In [ ]:
import os
import sys
sys.dont_write_bytecode = True

from dotenv import find_dotenv, load_dotenv
load_dotenv(find_dotenv())

import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)

# api keys
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
BASE_URL = os.getenv("BASE_URL")

# model selection
SMALL_MODEL_NAME = os.getenv("SMALL_MODEL_NAME")
LARGE_MODEL_NAME = os.getenv("LARGE_MODEL_NAME")
JUDGE_MODEL_NAME = os.getenv("JUDGE_MODEL_NAME")
SLM_AS_ROUTER = os.getenv("SLM_AS_ROUTER")

# concurency level
RPS = 5

# additional configuration
DATA_LOCATION_PATH = os.getenv("DATA_LOCATION_PATH")
TOKENIZER_NAME = os.getenv("TOKENIZER_NAME")
SPACY_NLP_MODEL = os.getenv("SPACY_NLP_MODEL")

# policies
SLM_AS_ROUTER_CONFIDENCE_THRESHOLD = 4.0

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

from core.data import RAGSyntheticDataset
from core.pipeline import RAGPipelineRunner, JScore
from core.messaging import LangchainMessageBuilder, PROMPT_REGISTRY
from core.router import (
    SLMRouterOutput,
    SLMRoutingPolicy,
    WeightedRuleBasedRoutingPolicy,
    WeightedRule
)
from core.tasks import RAGTaskPrediction
from core.utils import compute_slm_routing_metrics, dump_to_csv

In [ ]:
dataset = RAGSyntheticDataset.from_files(DATA_LOCATION_PATH)

In [ ]:
_base_rules = [
    WeightedRule(
        name="query_token_count",
        operator="",
        threshold=...,
        weight=...
    ),
    WeightedRule(
        name="query_noun_chunk_count",
        operator="",
        threshold=...,
        weight=...
    ),
    WeightedRule(
        name="query_avg_word_frequency",
        operator="",
        threshold=...,
        weight=...
    ),
    WeightedRule(
        name="avg_lexical_overlap",
        operator="",
        threshold=...,
        weight=...
    ),
]
reranking_task_rules = _base_rules + [
    WeightedRule(
        name="min_lexical_overlap",
        operator="",
        threshold=...,
        weight=...
    ),
    WeightedRule(
        name="documents_cosine_similarity",
        operator="",
        threshold=...,
        weight=...
    ),
    WeightedRule(
        name="documents_count",
        operator="",
        threshold=...,
        weight=...
    ),
]
compression_task_rules = _base_rules + [
    WeightedRule(
        name="total_context_token_count",
        operator="",
        threshold=...,
        weight=...
    ),
    WeightedRule(
        name="avg_chunk_token_count",
        operator="",
        threshold=...,
        weight=...
    ),
    WeightedRule(
        name="relevant_documents_ratio",
        operator="",
        threshold=...,
        weight=...
    ),
]
message_builder = LangchainMessageBuilder.from_sequence(
    ("reranking", PROMPT_REGISTRY.reranking_inference, RAGTaskPrediction),
    ("context_compression", PROMPT_REGISTRY.context_compression_inference, RAGTaskPrediction),
    ("judge", PROMPT_REGISTRY.evaluation, JScore)
)
model_kwargs = {
    "api_key": OPENROUTER_API_KEY,
    "base_url": BASE_URL,
    "max_tokens": 4092,
    "temperature": 0.0,
    "rate_limiter": InMemoryRateLimiter(
        requests_per_second=RPS,
        check_every_n_seconds=0.1,
        max_bucket_size=RPS * 2
    )
}

In [ ]:
small_model_only_pipeline = RAGPipelineRunner(
    small_model=SMALL_MODEL_NAME,
    large_model=LARGE_MODEL_NAME,
    judge_model=JUDGE_MODEL_NAME,
    routing_mode="slm",
    messages_builder=message_builder,
    extractor_spacy_nlp=SPACY_NLP_MODEL,
    extractor_tokenizer_name=TOKENIZER_NAME,
    model_kwargs=model_kwargs
)

In [ ]:
small_model_pipeline_result = await small_model_only_pipeline.arun(dataset)

In [ ]:
small_model_pipeline_evaluation = await small_model_only_pipeline.aevaluate(small_model_pipeline_result)

In [ ]:
large_model_only_pipeline = RAGPipelineRunner(
    small_model=SMALL_MODEL_NAME,
    large_model=LARGE_MODEL_NAME,
    judge_model=JUDGE_MODEL_NAME,
    routing_mode="llm",
    messages_builder=message_builder,
    extractor_spacy_nlp=SPACY_NLP_MODEL,
    extractor_tokenizer_name=TOKENIZER_NAME,
    model_kwargs=model_kwargs
)

In [ ]:
large_model_pipeline_result = await large_model_only_pipeline.arun(dataset)

In [ ]:
large_model_pipeline_evaluation = await large_model_only_pipeline.aevaluate(large_model_pipeline_result)

In [ ]:
rule_based_dynamic_routing_pipeline = RAGPipelineRunner(
    small_model=SMALL_MODEL_NAME,
    large_model=LARGE_MODEL_NAME,
    judge_model=JUDGE_MODEL_NAME,
    routing_mode="dynamic",
    messages_builder=message_builder,
    dynamic_routing_policies={
        "reranking": WeightedRuleBasedRoutingPolicy(
            *reranking_task_rules,
            min_triggers=2,
            cumulative_weights_threshold=...
        ),
        "context_compression": WeightedRuleBasedRoutingPolicy(
            *compression_task_rules,
            min_triggers=2,
            cumulative_weights_threshold=...
        )
    },
    extractor_spacy_nlp=SPACY_NLP_MODEL,
    extractor_tokenizer_name=TOKENIZER_NAME,
    model_kwargs=model_kwargs
)

In [ ]:
rule_based_pipeline_result = await rule_based_dynamic_routing_pipeline.arun(dataset)
dump_to_csv(rule_based_pipeline_result, path="rule_based_pipeline_result")

In [ ]:
rule_based_pipeline_evaluation = await rule_based_dynamic_routing_pipeline.aevaluate(rule_based_pipeline_result)
dump_to_csv(rule_based_pipeline_evaluation, path="rule_based_pipeline_evaluation")

In [ ]:
rule_based_pipeline_routing_metrics = compute_slm_routing_metrics(rule_based_pipeline_evaluation)
dump_to_csv(rule_based_pipeline_routing_metrics, path="rule_based_pipeline_routing_metrics")

In [ ]:
slm_router_client = ChatOpenAI(model=SLM_AS_ROUTER, **model_kwargs)

slm_routing_policy_messages_builder = LangchainMessageBuilder.from_sequence(
    ("slm_routing_policy", PROMPT_REGISTRY.slm_as_router, SLMRouterOutput)
)
slm_as_router_policy = SLMRoutingPolicy(
    client=slm_router_client,
    message_builder=slm_routing_policy_messages_builder,
    confidence_threshold=SLM_AS_ROUTER_CONFIDENCE_THRESHOLD
)
slm_router_based_pipeline = RAGPipelineRunner(
    small_model=SMALL_MODEL_NAME,
    large_model=LARGE_MODEL_NAME,
    judge_model=JUDGE_MODEL_NAME,
    routing_mode="dynamic",
    messages_builder=message_builder,
    dynamic_routing_policies={
        "reranking": slm_as_router_policy,
        "context_compression": slm_as_router_policy
    },
    extractor_spacy_nlp=SPACY_NLP_MODEL,
    extractor_tokenizer_name=TOKENIZER_NAME,
    model_kwargs=model_kwargs
)

In [ ]:
slm_router_based_pipeline_result = await slm_router_based_pipeline.arun(dataset)
dump_to_csv(slm_router_based_pipeline_result, path="slm_router_based_pipeline_result")

In [ ]:
slm_router_based_pipeline_evaluation = await slm_router_based_pipeline.aevaluate(slm_router_based_pipeline_result)
dump_to_csv(slm_router_based_pipeline_evaluation, path="slm_router_based_pipeline_evaluation")

In [ ]:
slm_router_based_pipeline_routing_metrics = compute_slm_routing_metrics(slm_router_based_pipeline_evaluation)
dump_to_csv(slm_router_based_pipeline_routing_metrics, path="slm_router_based_pipeline_routing_metrics")